In [1]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install curl_cffi

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
from curl_cffi import requests
from bs4 import BeautifulSoup
import time

In [11]:
query = "phone"
url1 = f"https://www.amazon.eg/s?k={query}"
url2 = f"https://www.jumia.com.eg/catalog/?q={query}"

In [12]:
session = requests.Session()

In [13]:
session.get("https://www.amazon.eg/", impersonate="chrome")
time.sleep(2)
amazon = session.get(url1, impersonate="chrome")
print("Amazon:", amazon.status_code)

Amazon: 200


In [14]:
session.get("https://www.jumia.com.eg/", impersonate="chrome")
time.sleep(2)
jumia = session.get(url2, impersonate="chrome")
print("Jumia:", jumia.status_code)

Jumia: 200


In [15]:
amazon.content

b'<!doctype html><html lang="en-ae" class="a-no-js" data-19ax5a9jf="dingo"><!-- sp:feature:head-start -->\n<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>\n<!-- sp:end-feature:head-start -->\n<!-- sp:feature:csm:head-open-part1 -->\n\n<script type=\'text/javascript\'>var ue_t0=ue_t0||+new Date();</script>\n<!-- sp:end-feature:csm:head-open-part1 -->\n<!-- sp:feature:cs-optimization -->\n<meta http-equiv=\'x-dns-prefetch-control\' content=\'on\'>\n<link rel="preconnect" href="https://images-eu.ssl-images-amazon.com" crossorigin>\n<link rel="preconnect" href="https://m.media-amazon.com" crossorigin>\n<!-- sp:end-feature:cs-optimization -->\n<!-- sp:feature:csm:head-open-part2 -->\n<script type=\'text/javascript\'>\nwindow.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;\nif (window.ue_ihb === 1) {\n\nvar ue_csm = window,\n    ue_hob = +new Date();\n(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()

In [16]:
jumia.content

b'<!DOCTYPE html><html lang="en" dir="ltr"><head><meta charset="utf-8"/><title>Shop All Products @ #1 Online Store - Enjoy Online Shopping @ Best Prices - Jumia Egypt</title><meta property="og:locale" content="en_EG"/><meta property="og:locale:alternate" content="ar_EG"/><meta name="title" content="Shop All Products @ #1 Online Store - Enjoy Online Shopping @ Best Prices - Jumia Egypt"/><meta name="robots" content="noindex,follow"/><meta name="description" content="Select Your Favorite Products from Jumia Egypt\'s Online Store - Start Online Shopping Today for Products of All Kinds! - Fast Delivery - Free Returns"/><meta property="og:type" content="website"/><meta property="og:title" content="Shop All Products @ #1 Online Store - Enjoy Online Shopping @ Best Prices - Jumia Egypt"/><meta property="og:site_name" content="Jumia Egypt"/><meta property="og:description" content="Select Your Favorite Products from Jumia Egypt\'s Online Store - Start Online Shopping Today for Products of All K

In [17]:
amazonSoup = BeautifulSoup(amazon.content, 'html.parser')
print(amazonSoup)

<!DOCTYPE html>
<html class="a-no-js" data-19ax5a9jf="dingo" lang="en-ae"><!-- sp:feature:head-start -->
<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>
<!-- sp:end-feature:head-start -->
<!-- sp:feature:csm:head-open-part1 -->
<script type="text/javascript">var ue_t0=ue_t0||+new Date();</script>
<!-- sp:end-feature:csm:head-open-part1 -->
<!-- sp:feature:cs-optimization -->
<meta content="on" http-equiv="x-dns-prefetch-control"/>
<link crossorigin="" href="https://images-eu.ssl-images-amazon.com" rel="preconnect"/>
<link crossorigin="" href="https://m.media-amazon.com" rel="preconnect"/>
<!-- sp:end-feature:cs-optimization -->
<!-- sp:feature:csm:head-open-part2 -->
<script type="text/javascript">
window.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;
if (window.ue_ihb === 1) {

var ue_csm = window,
    ue_hob = +new Date();
(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()-(b?0:d.ue_t0)};e.st

In [19]:
jumiaSoup = BeautifulSoup(jumia.content, 'html.parser')
print(jumiaSoup)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [37]:
amazon_records = []
amazonPhones = amazonSoup.find_all("div", class_="puis-card-container")

for item in amazonPhones:
    title_el = item.select_one("h2 span")
    price_el = item.select_one(".a-price-whole")
    rating_el = item.select_one("span.a-icon-alt")
    avail_el = item.select_one("span.a-color-price")
    
    amazon_records.append({
        "Title": title_el.get_text(strip=True) if title_el else None,
        "Price": price_el.get_text(strip=True) if price_el else None,
        "Rating": rating_el.get_text(strip=True) if rating_el else None,
        "Availability": avail_el.get_text(strip=True) if avail_el else "In Stock",
        "Source": "Amazon"
    })

df_amazon = pd.DataFrame(amazon_records)

In [39]:
jumia_records = []
jumiaPhones = jumiaSoup.find_all("article", class_="prd")

for item in jumiaPhones:
    title_el = item.select_one("h3.name")
    price_el = item.select_one("div.prc")
    rating_el = item.select_one("div.stars._s")
    
    jumia_records.append({
        "Title": title_el.get_text(strip=True) if title_el else None,
        "Price": price_el.get_text(strip=True) if price_el else None,
        "Rating": rating_el.get_text(strip=True) if rating_el else None,
        "Availability": "In Stock",
        "Source": "Jumia"
    })

df_jumia = pd.DataFrame(jumia_records)

In [40]:
df_raw = pd.concat([df_amazon, df_jumia], ignore_index=True)

display(df_raw.head(10))

df_raw.info()

display(df_raw.describe(include="all"))

print(df_raw.isnull().sum())

,Title,Price,Rating,Availability,Source
0,"HMD 100 Dual Sim Keypad Feature Mobile Phone, ...",785.,3.7 out of 5 stars,In Stock,Amazon
1,HMD 102 Dual SIM Keypad Feature Phone With Cam...,920.,3.0 out of 5 stars,In Stock,Amazon
2,HMD 100 TA-1753 Dual Sim Keypad Feature Mobile...,785.,None,In Stock,Amazon
3,HMD 102 Dual SIM Keypad Feature Phone With Cam...,920.,3.0 out of 5 stars,In Stock,Amazon
4,"HONOR X5c Plus 4G Smartphone, 4GB RAM 128GB RO...","8,250.",4.4 out of 5 stars,In Stock,Amazon
5,"XIAOMI REDMI 17 Smartphone, 6GB RAM + 256GB, 6...","11,999.",3.7 out of 5 stars,In Stock,Amazon
6,"HONOR X9d 5G Smartphone, 8GB RAM + 256GB Stora...","20,499.",4.6 out of 5 stars,In Stock,Amazon
7,"XIAOMI REDMI 17 Smartphone, 6GB RAM + 256GB, 6...","11,999.",3.7 out of 5 stars,In Stock,Amazon
8,"Samsung Galaxy A07s LTE, Android Smartphone, 6...","8,499.",3.8 out of 5 stars,In Stock,Amazon
9,"Infinix SMART 20 – Shadow Black ""LCD Display 6...","6,999.",4.4 out of 5 stars,In Stock,Amazon


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Title         100 non-null    object
 1   Price         100 non-null    object
 2   Rating        54 non-null     object
 3   Availability  100 non-null    object
 4   Source        100 non-null    object
dtypes: object(5)
memory usage: 4.0+ KB


,Title,Price,Rating,Availability,Source
count,100,100,54,100,100
unique,96,79,26,5,2
top,HMD 102 Dual SIM Keypad Feature Phone With Cam...,"EGP 9,999.00",3.0 out of 5 stars,In Stock,Amazon
freq,2,3,7,90,60


Title            0
Price            0
Rating          46
Availability     0
Source           0
dtype: int64


In [41]:
df_clean = df_raw.copy()

df_clean["Title"] = df_clean["Title"].str.strip().str.lower()

df_clean["Price"] = df_clean["Price"].str.replace(",", "").str.extract(r"(\d+\.?\d*)").astype(float)
df_clean["Rating"] = df_clean["Rating"].str.extract(r"(\d+\.\d+)").astype(float)

df_clean["Rating"] = df_clean["Rating"].fillna(df_clean["Rating"].mean().round(1))
df_clean = df_clean.dropna().drop_duplicates().reset_index(drop=True)

In [42]:
display(df_clean.head(10))
df_clean.info()
display(df_clean.describe())
print(df_clean.isnull().sum())
comparison = df_clean.groupby("Source")["Price"].agg(["count", "mean", "min", "max"]).round(2)
display(comparison)

,Title,Price,Rating,Availability,Source
0,"hmd 100 dual sim keypad feature mobile phone, ...",785.0,3.7,In Stock,Amazon
1,hmd 102 dual sim keypad feature phone with cam...,920.0,3.0,In Stock,Amazon
2,hmd 100 ta-1753 dual sim keypad feature mobile...,785.0,3.7,In Stock,Amazon
3,hmd 102 dual sim keypad feature phone with cam...,920.0,3.0,In Stock,Amazon
4,"honor x5c plus 4g smartphone, 4gb ram 128gb ro...",8250.0,4.4,In Stock,Amazon
5,"xiaomi redmi 17 smartphone, 6gb ram + 256gb, 6...",11999.0,3.7,In Stock,Amazon
6,"honor x9d 5g smartphone, 8gb ram + 256gb stora...",20499.0,4.6,In Stock,Amazon
7,"xiaomi redmi 17 smartphone, 6gb ram + 256gb, 6...",11999.0,3.7,In Stock,Amazon
8,"samsung galaxy a07s lte, android smartphone, 6...",8499.0,3.8,In Stock,Amazon
9,"infinix smart 20 – shadow black ""lcd display 6...",6999.0,4.4,In Stock,Amazon


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Title         96 non-null     object 
 1   Price         96 non-null     float64
 2   Rating        96 non-null     float64
 3   Availability  96 non-null     object 
 4   Source        96 non-null     object 
dtypes: float64(2), object(3)
memory usage: 3.9+ KB


,Price,Rating
count,96.000000,96.000000
mean,13417.687500,3.736458
std,13264.758113,0.654880
min,106.000000,1.000000
25%,6999.000000,3.700000
50%,10699.000000,3.700000
75%,16149.000000,4.025000
max,92833.000000,5.000000


Title           0
Price           0
Rating          0
Availability    0
Source          0
dtype: int64


,count,mean,min,max
Source,,,,
Amazon,56,13020.23,106.0,92833.0
Jumia,40,13974.12,6999.0,30799.0
